[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Creating and Changing Rows &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the catalog's models, the catalog loaded, and the counting
database the tasks measure with. Run it first. The tasks write to the table, so run them in order.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, CompositeKey, ForeignKeyField, IntegerField, Model, SqliteDatabase,
                    chunked)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

class CountingSqlite(SqliteDatabase):
    """A database that remembers how many statements went through it, and what the last one was."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.sent, self.last = 0, ""

    def execute_sql(self, sql, params=None):
        self.sent, self.last = self.sent + 1, " ".join(sql.split())
        return super().execute_sql(sql, params)

db = CountingSqlite(":memory:")


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
written = db.sent
print("peewee", peewee.__version__, "| the catalog:", Author.select().count(), "authors and",
      Book.select().count(), "books, written in", written, "statements")


peewee 4.5.1 | the catalog: 4 authors and 12 books, written in 8 statements


**1.** One new book, with the `id` the database gave it.


In [2]:
ursula = Author.get(Author.name == "Ursula Vance")

db.sent = 0
new = Book.create(title="The Ninth Wave", author=ursula, year=2024, pages=305)
print("id:", new.id, "| statements:", db.sent)
print("sent:", db.last)


id: 13 | statements: 1
sent: INSERT INTO "book" ("title", "author_id", "year", "pages") VALUES (?, ?, ?, ?)


The `id` was `None` until the `INSERT` ran, and `create` read it back off the cursor. That is the
reason to use `create` rather than `insert_many` for a row whose key another table will need.


**2.** An update that names one column.


In [3]:
held = Book.get(Book.title == "Nightjar")
held.pages = held.pages + 5

print("returned:", held.save(only=[Book.pages]))
print("sent:", db.last)


returned: 1
sent: UPDATE "book" SET "pages" = ? WHERE ("book"."id" = ?)


Without `only`, `save` writes every column, including the four that did not change. With it, the
`UPDATE` names `pages` and nothing else.


**3.** A key you fill in yourself, failing and then working.


In [4]:
class Genre(CatalogModel):
    name = CharField(primary_key=True)
    shelf = CharField(max_length=10)


db.create_tables([Genre])
crime = Genre(name="crime", shelf="C1")

returned = crime.save()
statement = db.last                                                 # before the count runs

print("save() returned:", returned, "| rows:", Genre.select().count())
print("sent:", statement)
print("force_insert returned:", crime.save(force_insert=True), "| rows:", Genre.select().count())


save() returned: 0 | rows: 0
sent: UPDATE "genre" SET "shelf" = ? WHERE ("genre"."name" = ?)
force_insert returned: 1 | rows: 1


The `UPDATE` went looking for a row that had never been written. Nothing was raised, and the only
sign was the `0`.


**4.** Five hundred rows in pieces of fifty.


In [5]:
batch = [{"title": f"Pamphlet {n}", "author": ursula, "year": 1990, "pages": 40} for n in range(500)]

db.sent = 0
with db.atomic():
    for piece in chunked(batch, 50):
        Book.insert_many(piece).execute()
sent = db.sent

print("statements:", sent, "| books now:", Book.select().count())
Book.delete().where(Book.title.startswith("Pamphlet")).execute()


statements: 10 | books now: 513


500

Ten statements for five hundred rows, and one transaction around all ten. A `create` loop would have
sent five hundred.


**5.** `get_or_create` twice, and the defaults it did not use.


In [6]:
first, created_first = Author.get_or_create(name="Halima Said", defaults={"first_book": 2011})
second, created_second = Author.get_or_create(name="Halima Said", defaults={"first_book": 1966})

print("first call  -> created:", created_first, "| first_book:", first.first_book)
print("second call -> created:", created_second, "| first_book:", second.first_book)


first call  -> created: True | first_book: 2011
second call -> created: False | first_book: 2011


`1966` was passed and `2011` came back, because the row existed and `defaults` is read only on the
create half. `on_conflict` is the call for a value that should be written either way.


**6.** Ten pages onto one author's books, in one statement.


In [7]:
change = (Book.update({Book.pages: Book.pages + 10})
              .where(Book.author == ursula))
print(sql(change))

db.sent = 0
rows = change.execute()
print("rows changed:", rows, "| statements:", db.sent)


UPDATE "book" SET "pages" = ("book"."pages" + ?) WHERE ("book"."author_id" = ?)  [10, 1]
rows changed: 4 | statements: 1


The arithmetic is in the SQL, so the pages were read, added to and written back without any of those
numbers reaching Python. The count is the number of rows the database changed, which is what makes
it worth printing.


---

&#8592; **Back to:** [Creating and Changing Rows](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/03-creating-and-changing-rows.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
